# P2-01 - Perbandingan model regresi `future_7d_mean_sbp`
**DATA SINTETIS / SIMULASI. Target belum disetujui dosen (`target_approved=false`). Bukan diagnosis dan bukan hasil penelitian aktual.**

Notebook ini melanjutkan notebook 02 (data readiness) dan 03 (verifikasi baseline CP3) tanpa menimpa artefak CP3 (`models/*.joblib`, `reports/cp3_baseline`). Artefak baru ditulis ke `models/experiments/` dan `reports/p2_model_comparison/`. Jalankan semua sel dari atas dengan kernel `.venv` project; notebook membaca dua CSV sintetis saja dan tidak membuka database.

In [1]:
from pathlib import Path
import sys, json
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from backend.app.pipeline import prepare
from backend.app.baseline import make_training, partitions, FEATURES, TARGET
from backend.app.model_comparison import (run_comparison, validate, select_model, load_comparison, column_key, HYPERPARAMETERS, NAIVE, LEARNED, SEED)
from etl.run_model_comparison import write_figures
profile = root / 'data/sample/Data_Responden_Sintetis.csv'
daily = root / 'data/sample/Data_Harian_Sintetis.csv'
print('Target:', TARGET, '| seed:', SEED, '| fitur:', FEATURES)

Target: future_7d_mean_sbp | seed: 42 | fitur: ['past_7d_sbp', 'past_7d_dbp', 'past_7d_sbp_std', 'age', 'bmi', 'past_7d_activity', 'past_7d_sleep', 'past_7d_stress']


## 1. Pemilihan algoritma (alasan, bukan sekadar menambah jumlah)
Data berukuran kecil (puluhan window train), fitur numerik tabular, dan sebagian fitur berkorelasi. Karena itu kandidat dipilih agar mewakili tiga keluarga model:
1. **Baseline naif** - rerata SBP 7 hari terakhir. Pembanding wajib: model terlatih hanya berguna bila lebih baik dari ini.
2. **Linear Regression** - sederhana, mudah dijelaskan, varians rendah.
3. **Random Forest Regressor** - non-linear berbasis bagging; tahan outlier; hiperparameter dibatasi (`max_depth`, `min_samples_leaf`) karena data kecil.
4. **HistGradientBoostingRegressor** - boosting non-linear; learning rate rendah dan pohon dangkal agar tidak overfit.

Hiperparameter tetap, tidak dituning pada test.

In [2]:
display(pd.DataFrame({k: [json.dumps(v, sort_keys=True)] for k, v in HYPERPARAMETERS.items()}, index=['hiperparameter']).T)

,hiperparameter
Linear Regression,{}
Random Forest,"{""max_depth"": 5, ""max_features"": 0.6, ""min_sam..."
Hist Gradient Boosting,"{""l2_regularization"": 1.0, ""learning_rate"": 0...."


## 2. Window, split, dan pencegahan leakage
Origin = akhir hari t; fitur dari histori t-6..t; target = rerata SBP harian t+1..t+7. Split train/test **identik** dengan baseline CP3: ID held-out, purge label kronologis (cutoff 60% kalender dipilih sebelum melihat skor), dan target uji tidak overlap. Imputer median dan scaler di-fit pada data fit saja.

In [3]:
p, d, s, quality = prepare(pd.read_csv(profile, dtype=str, keep_default_na=False), pd.read_csv(daily, dtype=str, keep_default_na=False), synthetic=True)
frame = make_training(p, d, s)
parts = partitions(frame)
print('Seluruh window:', len(frame), '| train:', len(parts['train']), '| test utama:', len(parts['test']), '| cutoff:', parts['cutoff'])
print('ID train:', parts['train'].respondent_id.nunique(), '| ID test:', parts['test'].respondent_id.nunique())
print('ID bersama train-test:', set(parts['train'].respondent_id) & set(parts['test'].respondent_id))

Seluruh window: 323 | train: 60 | test utama: 7 | cutoff: 2026-09-18
ID train: 12 | ID test: 7
ID bersama train-test: set()


## 3. Validasi model (hanya di dalam train)
Pemilihan model memakai **leave-ID-out GroupKFold** pada train. Test belum disentuh. Aturan pemilihan dikunci: RMSE validasi terkecil di antara model terlatih (tie -> MAE).

In [4]:
val = validate(parts['train'])
table = pd.DataFrame(val['scores']).T[['n', 'mae', 'rmse', 'r2']]
display(table.round(3))
selection = select_model(val['scores'])
print('Dipilih (berdasarkan validasi):', selection['selected'], '| mengalahkan naif pada validasi:', selection['beats_naive_on_validation'])
display(pd.DataFrame(val['folds']))

,n,mae,rmse,r2
Naive 7-day mean,60.0,1.908,2.377,0.865
Linear Regression,60.0,2.180,2.759,0.818
Random Forest,60.0,3.178,4.750,0.459
Hist Gradient Boosting,60.0,3.353,4.600,0.493


Dipilih (berdasarkan validasi): Linear Regression | mengalahkan naif pada validasi: False


,fold,fit_rows,validation_rows,validation_ids
0,0,45,15,"[SIM008, SIM014, SIM019]"
1,1,45,15,"[SIM007, SIM013, SIM018]"
2,2,45,15,"[SIM005, SIM011, SIM016]"
3,3,45,15,"[SIM003, SIM010, SIM015]"


## 4. Eksekusi penuh: fit, audit leakage, test sekali, simpan artefak
`run_comparison` melakukan urutan yang sama, menjalankan `leakage_audit` CP3 untuk setiap model, menghitung test **sekali**, lalu menulis metrik, manifest berhash, model terpilih, dan `MODEL_CARD.md`.

In [5]:
result = run_comparison(profile, daily, root / 'models/experiments', root / 'reports/p2_model_comparison')
write_figures(result, root / 'reports/p2_model_comparison')
rows = [dict(model=c['model'], val_rmse=c['validation']['rmse'], val_mae=c['validation']['mae'], test_rmse=c['test']['rmse'], test_mae=c['test']['mae'], test_r2=c['test']['r2']) for c in result['comparison']]
display(pd.DataFrame(rows).round(3))
print('Run:', result['run_id'], '| terpilih:', result['selected_model'])
for w in result['warnings']: print('PERINGATAN:', w)

,model,val_rmse,val_mae,test_rmse,test_mae,test_r2
0,Naive 7-day mean,2.377,1.908,2.235,1.934,0.850
1,Linear Regression,2.759,2.180,3.114,2.575,0.709
2,Random Forest,4.750,3.178,3.441,2.959,0.645
3,Hist Gradient Boosting,4.600,3.353,4.832,3.693,0.300


Run: 44ce7fdaf09c5de804ff | terpilih: Linear Regression
PERINGATAN: Test hanya 7 origin: metrik test sangat tidak stabil.
PERINGATAN: Train hanya 60 window: model non-linear mudah overfit.
PERINGATAN: Tidak ada model terlatih yang mengalahkan baseline naif pada validasi.


In [6]:
display(pd.Series(result['leakage_audit']))
assert result['target_approved'] is False and result['is_synthetic']

subject_overlap                                []
shared_measurement_id_dates                     0
train_target_end_max                   2026-09-18
test_origin_min                        2026-09-18
labels_available_before_test                 True
test_overlapping_target_id_days                 0
imputer_matches_training_only                True
scaler_matches_training_only                 True
training_overlapping_target_id_days           288
dtype: object

## 5. Aktual vs prediksi (test utama)

In [7]:
pred = pd.read_csv(root / 'reports/p2_model_comparison/test_predictions.csv')
fig, ax = plt.subplots(figsize=(6, 5))
for name in [NAIVE, *LEARNED]:
    ax.scatter(pred.actual, pred[column_key(name)], label=name, alpha=.7)
lo, hi = pred.actual.min() - 3, pred.actual.max() + 3
ax.plot([lo, hi], [lo, hi], 'k--', alpha=.5); ax.set(xlabel='Aktual (mmHg)', ylabel='Prediksi (mmHg)', title='SINTETIS | n=%d origin' % len(pred)); ax.legend(fontsize=8); ax.grid(alpha=.2)
plt.show()

## 6. Kesimpulan dan batasan (dibaca dari hasil, bukan ditulis manual)

In [8]:
by = {c['model']: c for c in result['comparison']}
naive = by[NAIVE]
for name in LEARNED:
    c = by[name]
    verdict = 'LEBIH BAIK' if c['validation']['rmse'] < naive['validation']['rmse'] else 'tidak lebih baik'
    print(f"{name:<24} RMSE validasi {c['validation']['rmse']:.3f} vs naif {naive['validation']['rmse']:.3f} -> {verdict} dari baseline naif")
print()
print(result['limitation'])
print('\nStatus:', result['status'])

Linear Regression        RMSE validasi 2.759 vs naif 2.377 -> tidak lebih baik dari baseline naif
Random Forest            RMSE validasi 4.750 vs naif 2.377 -> tidak lebih baik dari baseline naif
Hist Gradient Boosting   RMSE validasi 4.600 vs naif 2.377 -> tidak lebih baik dari baseline naif

DATA SINTETIS. Train hanya 60 window dari 12 ID dan test utama hanya 7 origin dari 7 ID, sehingga selisih antar-model tidak stabil dan tidak membuktikan generalisasi. Window train saling berkorelasi. Target belum disetujui dosen. Bukan validasi klinis atau diagnosis.

Status: DATA SINTETIS / SIMULASI - EKSPERIMEN


### Pembacaan
- Hasil ini hanya menguji alur teknis pada data sintetis; selisih antar-model **tidak** membuktikan model mana yang lebih baik pada responden aktual.
- Bila baseline naif tidak terkalahkan pada validasi, itu temuan sah: model terlatih belum memberi nilai tambah pada data sekecil ini.
- Test utama sangat kecil (7 origin) sehingga tidak boleh dipakai untuk memilih ulang model.
- Langkah lanjut menunggu persetujuan target oleh dosen dan validasi ekspor data aktual.